In [1]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.preprocessing import PolynomialFeatures
import pandas as pd
import pickle
import numpy as np

# Load datasets
specific = pd.read_csv("specific.csv")
alldata = pd.read_csv("alldata.csv")
teamdata = pd.read_csv("teamdata.csv")

# Clean keys
for d in [specific, alldata, teamdata]:
    d["map"] = d["map"].astype(str).str.strip()

# Calculate global ban rate from alldata only
alldata["global_ban_rate"] = alldata["ban"] / alldata["ban"].sum()
alldata["global_pick_rate"] = alldata["pick"] / alldata["pick"].sum()

# Calculate team pick rate from teamdata
teamdata["team_pick_rate"] = teamdata["pick"] / teamdata["pick"].sum()
teamdata["team_ban_rate"] = teamdata["ban"] / teamdata["ban"].sum()

# Merge all rates into one table on map
merged = (
    alldata[["map", "global_ban_rate", "global_pick_rate"]]
    .merge(teamdata[["map", "team_pick_rate", "team_ban_rate"]], on="map", how="inner")
    .merge(specific[["map", "pick"]], on="map", how="inner")
)
merged["team"] = teamdata["team"].iloc[0]

# Features: global ban rate + team pick rate | Target: team specific pick count
X = merged[["global_ban_rate", "team_pick_rate"]].values
y = merged["pick"].values

# Polynomial features (degree 2)
poly = PolynomialFeatures(degree=2)
X_poly = poly.fit_transform(X)

pick_model = LinearRegression()
pick_model.fit(X_poly, y)

y_pred = pick_model.predict(X_poly)

results = merged[["team", "map", "global_ban_rate", "team_pick_rate", "pick"]].copy()
results["predicted_picks"] = y_pred
results["pick_probability"] = y_pred / y_pred.sum()
results["actual_probability"] = y / y.sum()

print("\nMap Selection Probabilities:")
print(results.to_string(index=False))

best_row = results.loc[results["pick_probability"].idxmax()]
worst_row = results.loc[results["pick_probability"].idxmin()]

print(
    f"\nBest probability map:   {best_row['map']} ({best_row['pick_probability']:.2%})"
)
print(
    f"Lowest probability map: {worst_row['map']} ({worst_row['pick_probability']:.2%})"
)

r2 = r2_score(y, y_pred)
accuracy_pct = max(r2 * 100, 0)
print(f"\nModel accuracy (R² score): {accuracy_pct:.1f}%")

filename = "map_selection_model.sav"
pickle.dump((poly, pick_model), open(filename, "wb"))
print(f"Model saved as: {filename}")


Map Selection Probabilities:
     team      map  global_ban_rate  team_pick_rate  pick  predicted_picks  pick_probability  actual_probability
Sentinels   Ascent         0.114194        0.000000     0       -73.758707         -0.019669            0.000000
Sentinels     Bind         0.084845        0.000000     0        63.491142          0.016931            0.000000
Sentinels  Corrode         0.060832        0.085714   183       266.525505          0.071073            0.048800
Sentinels Fracture         0.084312        0.028571    70       103.787916          0.027677            0.018667
Sentinels    Haven         0.136606        0.028571   150       235.566977          0.062818            0.040000
Sentinels   Icebox         0.115261        0.000000     0       -73.253538         -0.019534            0.000000
Sentinels    Lotus         0.120064        0.314286  2057      1958.457068          0.522255            0.548533
Sentinels    Pearl         0.095518        0.257143   594       82